file and import set up

In [2]:
import json
import fitz
import pandas as pd
import os
import spacy
print(os.getcwd())
main_path = "/Users/jaceysimpson/Vscode/FellowScript/"
bible_dict_path = os.path.join(main_path, "data/bible.json")
pdf_path = os.path.join(main_path, "data/ESV Bible.pdf")
book_list_path = os.path.join(main_path, "data/list_esv.csv")

doc = fitz.open(pdf_path)
nlp = spacy.load("en_core_web_sm")


/Users/jaceysimpson/Vscode/FellowScript/api/backend/bibleHandling


process PDF into lines

In [3]:
def process_pdf(out_file="data/list_esv.csv"):
    out_file = os.path.join(main_path, out_file)
    lines = []
    for page in doc:
        text = str(page.get_text())
        lines.extend(text.splitlines())
    df = pd.DataFrame({"lines": lines})
    df.to_csv(out_file)

process_pdf()

search book start and end

In [39]:
def get_start(start_idx, next_line, book_lines):
    idx = start_idx
    while "chapter" in next_line.lower() and idx < len(book_lines):
        idx += 1
        next_line = book_lines[idx]
    start = idx
    return start

def get_end(start_idx, next_book, book_lines):
    idx = start_idx
    end = 0
    while next_book.lower() not in book_lines[idx].lower() and idx < len(book_lines):
        idx += 1
    end = idx
    return end


search for book

In [ ]:
def find_book(book: str, book_lines: pd.Series):
        idx = 0
        start, end = (0, 0)
        print(f"num lines in df: {len(book_lines)}")
        while idx < len(book_lines):
            line = book_lines.iloc[idx]
            if book.lower() in line.lower():
                if idx + 1 > len(book_lines) - 1:
                    idx += 1
                    continue
                next_line = book_lines[idx+1]
                start_idx = idx
                if "chapter" in next_line.lower():
                    start = get_start(start_idx, next_line, book_lines)
                    end = get_end(start, "footnotes", book_lines)
                    break
            idx += 1
                    
        return (start, end)

book = "genesis"
book_lines = os.path.join(main_path, "data/list_esv.csv")
df = pd.read_csv(book_lines)
find_book(book, df["lines"])

num lines in df: 98407


(620, 4705)

remove headers

In [35]:
def remove_stops(line):
    doc = nlp(line)
    filtered = [word.text for word in doc if not word.is_stop]
    return filtered

def is_header(line):
    words = remove_stops(line)
    is_header = True
    if len(words) < 2:
        return False
    for word in words:
        if not word[0].isupper() or word.isdigit():
            is_header = False
    return is_header

line = "The Creation of the World"
print(is_header(line))

True


parse the chapter

In [ ]:
def new_chapter(line):
    if ":" in line:
        print(f"potential chapter: {line}")
        colon_idx = line.index(":")
        if colon_idx < len(line) - 1:
            if line[0].isdigit() and line[colon_idx+1].isdigit():
                print(f"line proves chapter".upper())
                return True
            else:
                print(f"line failed to prove as chapter".upper())
        else:
            print("line failed to prove as chapter".upper())
    return False

def parse_chapters(start: int, end: int, book_lines: pd.Series):
    idx = start
    chapters = []
    chapter = ""
    while idx < end:
        line = book_lines.loc[idx]
        if is_header(line):
            header = "HEAD::", line
            chapters.append(header)
            idx += 1
            continue
        if new_chapter(line) and len(chapter) > 0:
            chapters.append(chapter)
            chapter = line
        else:
            chapter += f" {line}"
        
        idx += 1
    chapters.append(chapter)
    return chapters
book = 'exodus'
book_lines = os.path.join(main_path, "data/list_esv.csv")
df = pd.read_csv(book_lines)
start, end = find_book(book, df["lines"])
parse_chapters(start, end, df["lines"]) #type: ignore

num lines in df: 98407
potential chapter: 1:1 These are the names of the sons of Israel who
LINE PROVES CHAPTER
potential chapter: came to Egypt with Jacob, each with his household:
LINE FAILED TO PROVE AS CHAPTER
potential chapter: 2:1 Now a man from the house of Levi went and
LINE PROVES CHAPTER
potential chapter: 3:1 Now Moses was keeping the flock of his father-
LINE PROVES CHAPTER
potential chapter: have sent you: when you have brought the people out
LINE FAILED TO PROVE AS CHAPTER
potential chapter: 4:1 Then Moses answered, “But behold, they will not
LINE PROVES CHAPTER
potential chapter: 5:1 Afterward Moses and Aaron went and said to
LINE PROVES CHAPTER
potential chapter: 6:1 But the LORD said to Moses, “Now you shall
LINE PROVES CHAPTER
potential chapter: people of Israel and about Pharaoh king of Egypt: to
LINE FAILED TO PROVE AS CHAPTER
potential chapter: 14These are the heads of their fathers' houses: the
LINE FAILED TO PROVE AS CHAPTER
potential chapter: sons of Reuben, the

[('HEAD::', 'Israel Increases Greatly in Egypt'),
 ('HEAD::', 'Pharaoh Oppresses Israel'),
 ('HEAD::', 'The Birth of Moses'),
 ' 1:1 These are the names of the sons of Israel who came to Egypt with Jacob, each with his household: 2Reuben, Simeon, Levi, and Judah, 3Issachar, Zebulun, and Benjamin, 4Dan and Naphtali, Gad and Asher. 5All the descendants of Jacob were seventy persons; Joseph was already in Egypt. 6Then Joseph died, and all his brothers and all that generation. 7But the people of Israel were fruitful and increased greatly; they multiplied and grew exceedingly strong, so that the land was filled with them. 8Now there arose a new king over Egypt, who did not know Joseph. 9And he said to his people, “Behold, the people of Israel are too many and too mighty for us. 10Come, let us deal shrewdly with them, lest they multiply, and, if war breaks out, they join our enemies and fight against us and escape from the land.” 11Therefore they set taskmasters over them to afflict them wit

pattern search for start of new chapter